# Clase 7 · Práctica: Deep learning para lenguaje natural

## Qué vas a practicar

En esta libreta aplicas lo que viste en la Clase 7 sobre procesamiento del lenguaje natural (NLP, del inglés Natural Language Processing) con deep learning. Completas el código en dos ejercicios:

- **Ejercicio 1.** Construyes un clasificador de opiniones de personas clientas (positiva o negativa) sobre un pequeño conjunto de frases de ejemplo, combinando una capa `nn.Embedding` con una capa lineal.
- **Ejercicio 2.** Reutilizas ese modelo para predecir el sentimiento de frases nuevas, escritas por ti, que incluyen palabras que el modelo nunca vio durante el entrenamiento.

Cada ejercicio incluye el andamiaje de código con partes marcadas como `# TODO`, que debes completar tú. Debajo de cada ejercicio encuentras una solución colapsada: intenta resolver el ejercicio por tu cuenta antes de abrirla.

## Requisito previo

Necesitas haber visto la Clase 7 ("Deep learning para lenguaje natural"), en la que se explica cómo se convierte el texto en números mediante tokenización y vocabulario, qué es un embedding y cómo `torch.nn.Embedding` lo implementa como una tabla entrenable.

## Cómo ejecutar esta libreta

Ábrela en Google Colab y ejecuta las celdas en orden, de arriba a abajo. No instales nada adicional: `torch` ya viene preinstalado en Colab. Este ejercicio entrena en pocos segundos, incluso sin GPU (unidad de procesamiento gráfico).

## Ejercicio 1 · Clasificador de opiniones de personas clientas

### El caso

Trabajas en el equipo de datos de una empresa que vende conservas gallegas a través de su tienda online. Cada semana llegan decenas de opiniones de personas clientas y quieres clasificarlas automáticamente como positivas o negativas, para priorizar la revisión manual de las quejas más urgentes. Como primera aproximación, entrenas un clasificador minúsculo con un puñado de opiniones de ejemplo escritas a mano.

### Lo que tienes que hacer

Todo el andamiaje (los datos, la construcción del vocabulario, la tokenización y el bucle de entrenamiento) ya está escrito. Tu tarea es completar la clase `ClasificadorOpiniones`: defines la capa de embedding y la capa lineal de clasificación en `__init__`, y escribes el cálculo hacia delante (`forward`), incluyendo cómo combinas los embeddings de las distintas palabras de una frase en un único vector.

### Pistas

- `nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)` crea la tabla de embeddings: una fila por palabra del vocabulario. `padding_idx=0` evita que el token `<PAD>` (relleno) reciba gradiente, porque no aporta significado.
- La capa de clasificación es una `nn.Linear` que recibe un vector de tamaño `dim_embedding` y produce un único logit (la entrada sin normalizar de la función sigmoide): cada frase es una única muestra con una única probabilidad de ser positiva.
- En `forward`, `self.embedding(x)` devuelve un tensor de forma `(lote, longitud_max, dim_embedding)`: un vector por cada palabra de cada frase del lote.
- Para obtener un único vector por frase, promedia los embeddings de sus palabras a lo largo del eje de longitud, con `.mean(dim=1)`. Es una simplificación deliberada: pierde el orden de las palabras, pero basta para este ejemplo.
- Al final, aplana el logit con `.squeeze(1)` para que tenga forma `(lote,)` en lugar de `(lote, 1)`: así coincide con la forma que espera `nn.BCEWithLogitsLoss`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- Datos de entrenamiento: opiniones minúsculas con su etiqueta ---
# Etiqueta 1 = opinión positiva, 0 = opinión negativa
frases = [
    "el producto llego en perfecto estado y sabe genial",
    "atencion al cliente rapida y muy amable",
    "la calidad de las conservas es excelente",
    "repetire compra sin duda todo perfecto",
    "envio rapido y muy bien empaquetado",
    "estoy encantada con el sabor y la textura",
    "precio justo para una calidad tan buena",
    "el equipo resolvio mi duda enseguida y con amabilidad",
    "paquete danado y producto en mal estado",
    "tardaron semanas en responder a mi reclamacion",
    "el sabor es soso y decepcionante",
    "no volvere a comprar pesima experiencia",
    "el envio llego tarde y sin avisar",
    "atencion al cliente lenta y poco amable",
    "la calidad no corresponde con el precio",
    "producto roto al abrir la caja",
]
etiquetas = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

# --- Construimos el vocabulario a mano ---
# <PAD> rellena frases cortas para igualar longitudes dentro de un lote.
# <UNK> representa cualquier palabra no vista durante el entrenamiento.
palabras = sorted(set(palabra for frase in frases for palabra in frase.split()))
vocabulario = ["<PAD>", "<UNK>"] + palabras
indice_de = {palabra: i for i, palabra in enumerate(vocabulario)}
TAMANO_VOCAB = len(vocabulario)
print(f"Tamaño del vocabulario: {TAMANO_VOCAB} palabras")


def tokenizar(frase, longitud_max):
    """Convierte una frase en una lista de índices, rellenada con <PAD>
    hasta longitud_max."""
    indices = [indice_de.get(p, indice_de["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_de["<PAD>"]] * (longitud_max - len(indices))
    return indices


LONGITUD_MAX = max(len(f.split()) for f in frases)
X = torch.tensor([tokenizar(f, LONGITUD_MAX) for f in frases], dtype=torch.long)
y = torch.tensor(etiquetas, dtype=torch.float32)
print(f"Forma de X: {X.shape}")  # (num_frases, LONGITUD_MAX)

In [ ]:
class ClasificadorOpiniones(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()

        # TODO: define la capa de embedding.
        # Debe tener tamano_vocab filas (una por palabra del vocabulario)
        # y dim_embedding columnas. Usa padding_idx=0 para que el token
        # <PAD> no reciba gradiente.
        self.embedding = ...

        # TODO: define la capa de clasificación.
        # Recibe un vector de tamaño dim_embedding (el promedio de los
        # embeddings de la frase) y produce un único logit.
        self.clasificador = ...

    def forward(self, x):
        # x llega con forma (lote, longitud_max): un índice por palabra

        # TODO: obtén los embeddings de x.
        # Forma resultante: (lote, longitud_max, dim_embedding)
        vectores = ...

        # TODO: promedia los embeddings de cada frase a lo largo del eje
        # de longitud (dim=1), para obtener un único vector por frase.
        # Forma resultante: (lote, dim_embedding)
        promedio = ...

        # TODO: pasa el promedio por la capa de clasificación y aplana
        # el resultado con .squeeze(1) para que tenga forma (lote,)
        logit = ...

        return logit


DIM_EMBEDDING = 8
modelo = ClasificadorOpiniones(TAMANO_VOCAB, DIM_EMBEDDING)
print(modelo)

In [ ]:
# --- Entrenamos ---
# Esta parte ya está resuelta: solo necesitas ejecutarla, una vez tu
# modelo del apartado anterior calcule el hacia delante correctamente.
perdida_fn = nn.BCEWithLogitsLoss()  # combina sigmoide + entropía cruzada, más estable
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

for epoca in range(200):
    optimizador.zero_grad()
    logits = modelo(X)
    perdida = perdida_fn(logits, y)
    perdida.backward()
    optimizador.step()
    if (epoca + 1) % 50 == 0:
        print(f"Época {epoca + 1}: pérdida = {perdida.item():.4f}")

<details>
<summary>💡 Ver solución</summary>

```python
class ClasificadorOpiniones(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()

        # Tabla de embeddings: tamano_vocab filas x dim_embedding columnas.
        # padding_idx=0 evita que <PAD> reciba gradiente: no aporta significado.
        self.embedding = nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)

        # Capa lineal que reduce el vector de la frase a un único logit
        self.clasificador = nn.Linear(dim_embedding, 1)

    def forward(self, x):
        vectores = self.embedding(x)         # (lote, longitud_max, dim_embedding)
        promedio = vectores.mean(dim=1)       # (lote, dim_embedding)
        logit = self.clasificador(promedio)   # (lote, 1)
        return logit.squeeze(1)               # (lote,)


DIM_EMBEDDING = 8
modelo = ClasificadorOpiniones(TAMANO_VOCAB, DIM_EMBEDDING)
print(modelo)
```
</details>

## Ejercicio 2 · Predice el sentimiento de opiniones nuevas

### El caso

El modelo del Ejercicio 1 ya entrena y clasifica bien las frases del conjunto de ejemplo, pero para que sea útil en el día a día necesitas aplicarlo a opiniones reales que nunca ha visto, escritas con palabras que no están en su vocabulario. Tu compañera de equipo va a integrar el modelo en un panel interno y te pide una función que reciba cualquier frase y devuelva si la considera positiva o negativa.

### Lo que tienes que hacer

Completa la función `predecir_sentimiento(modelo, frase)`. Convierte la frase en índices palabra por palabra, gestiona las palabras que no están en `vocabulario` con el token `<UNK>`, y ajusta la longitud de la secuencia igual que hace `tokenizar` con las frases de entrenamiento.

### Pistas

- `indice_de` es el diccionario que mapea cada palabra de `vocabulario` a su índice. Usa `indice_de.get(palabra, valor_por_defecto)` en lugar de `indice_de[palabra]`: así, si la palabra no existe en el diccionario, obtienes el valor por defecto en lugar de un error `KeyError`.
- El valor por defecto para una palabra desconocida es `indice_de["<UNK>"]`.
- Después de convertir la frase a índices, recórtala o rellénala hasta `LONGITUD_MAX`, con el mismo criterio que usa `tokenizar`: recorta con `indices[:LONGITUD_MAX]` y rellena con `indice_de["<PAD>"]` las posiciones que falten.
- El modelo espera un tensor de forma `(lote, longitud_max)`. Como predices sobre una única frase, envuelve la lista de índices en otra lista antes de convertirla a tensor: `torch.tensor([indices], dtype=torch.long)`.

In [ ]:
def predecir_sentimiento(modelo, frase):
    """Recibe una frase nueva escrita por el usuario y devuelve una
    tupla (etiqueta, probabilidad), donde etiqueta es "positiva" o
    "negativa". La frase puede contener palabras fuera del vocabulario
    de entrenamiento: esas palabras se tratan con el token <UNK>."""
    modelo.eval()

    # TODO: convierte cada palabra de `frase` en su índice del
    # vocabulario, usando indice_de.get(...) para asignar el índice de
    # "<UNK>" a las palabras que no estén en el vocabulario.
    indices = ...

    # TODO: ajusta `indices` a longitud LONGITUD_MAX: recórtalo si es
    # más largo, y rellénalo con el índice de "<PAD>" si es más corto.
    indices = ...

    entrada = torch.tensor([indices], dtype=torch.long)
    with torch.no_grad():
        probabilidad = torch.sigmoid(modelo(entrada)).item()

    etiqueta = "positiva" if probabilidad > 0.5 else "negativa"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")
    return etiqueta, probabilidad


predecir_sentimiento(modelo, "el servicio postventa fue estupendo")
predecir_sentimiento(modelo, "producto defectuoso y atencion pesima")
predecir_sentimiento(modelo, "muy contenta con la compra repetire seguro")

<details>
<summary>💡 Ver solución</summary>

```python
def predecir_sentimiento(modelo, frase):
    """Recibe una frase nueva escrita por el usuario y devuelve una
    tupla (etiqueta, probabilidad), donde etiqueta es "positiva" o
    "negativa". La frase puede contener palabras fuera del vocabulario
    de entrenamiento: esas palabras se tratan con el token <UNK>."""
    modelo.eval()

    # Cada palabra se convierte en su índice; si no está en el
    # vocabulario, se usa el índice de <UNK>
    indices = [indice_de.get(palabra, indice_de["<UNK>"]) for palabra in frase.split()]

    # Recorta o rellena hasta LONGITUD_MAX, igual que hace tokenizar()
    indices = indices[:LONGITUD_MAX]
    indices += [indice_de["<PAD>"]] * (LONGITUD_MAX - len(indices))

    entrada = torch.tensor([indices], dtype=torch.long)
    with torch.no_grad():
        probabilidad = torch.sigmoid(modelo(entrada)).item()

    etiqueta = "positiva" if probabilidad > 0.5 else "negativa"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")
    return etiqueta, probabilidad


predecir_sentimiento(modelo, "el servicio postventa fue estupendo")
predecir_sentimiento(modelo, "producto defectuoso y atencion pesima")
predecir_sentimiento(modelo, "muy contenta con la compra repetire seguro")
```
</details>

## Para terminar

Acabas de construir, con tus propias manos, la pieza central que convierte palabras en vectores, y de aplicarla a frases que el modelo nunca vio durante el entrenamiento. Comprobaste también por qué el promedio de embeddings, aunque funciona, ignora el orden de las palabras: es exactamente la limitación de la que parte el mecanismo de atención que verás en la próxima clase.

Antes de continuar, prueba a cambiar `DIM_EMBEDDING` a un valor mucho menor (por ejemplo, dos) y a uno mucho mayor (por ejemplo, treinta y dos), y observa cómo cambian la velocidad de entrenamiento y la pérdida final. Después, sustituye `vectores.mean(dim=1)` por una capa `nn.LSTM` y comprueba si el modelo distingue mejor frases como "el servicio no fue amable" frente a "el servicio fue amable".

**¡Comparte tus resultados y tus dudas en el canal del curso antes de la próxima clase en vivo!**